# READ ME FIRST - Quick Usage Guide

Use this notebook to create and expand a domain-specific LLM red-team dataset.

**Run order:** run every cell from top to bottom.

**Most users only change one thing:** in section 2, choose the config file:

- `configs/finance_benchmark.yaml` = safe local deterministic generation, no LLM API call.
- `configs/finance_benchmark.deepteam_llm.yaml` = live DeepTeam LLM generation, requires API key in `.env`.

**Main output files:**

- `data/exports/finance_redteam_attacks.jsonl` = final benchmark dataset.
- `data/exports/promptfoo_tests.yaml` = Promptfoo evaluation config.

**Important:** do not paste API keys into this notebook. Put them in `.env`.

**After this notebook:** open `02_evaluate_attacks_gemini.ipynb` to evaluate the generated attacks against Gemini.

# 01 - Create and Expand a Domain Red-Team Dataset

This notebook builds the benchmark dataset from scratch.

**Who should use this:** use this notebook when you want to create a domain-specific red-team prompt dataset, optionally expand it with DeepTeam/Garak, and export files that can be evaluated later with Promptfoo.

**Important safety note:** this project is for defensive model evaluation only. The generated prompts are test inputs that should make the target model refuse unsafe requests, protect sensitive data, and redirect to compliant guidance.

It is designed as a reusable utility notebook. The current project implementation is finance-specific, but the notebook keeps the domain settings in one place so you can adapt the workflow for another domain later by swapping taxonomy, seed prompts, and mappings.

Pipeline:

```text
Domain config -> taxonomy -> seed prompts -> local records -> DeepTeam expansion -> Garak patterns -> normalize -> deduplicate -> validate -> export JSONL + Promptfoo YAML
```

Default output:

```text
data/exports/finance_redteam_attacks.jsonl
data/exports/promptfoo_tests.yaml
```

**How to use this notebook:** run cells from top to bottom. The only cells you normally edit are the configuration cell and, if needed, the DeepTeam/Garak selector values in the YAML config files.

## 1. Setup

Run this notebook from the project root, or let the cell below locate the project root automatically.

This cell only sets paths. It does not generate data, call APIs, or modify project files.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "finance_redteam").exists():
    candidates = [p for p in Path.cwd().parents if (p / "src" / "finance_redteam").exists()]
    if not candidates:
        raise RuntimeError("Could not find project root. Open this notebook from finance-llm-redteam-benchmark.")
    PROJECT_ROOT = candidates[0]

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

## 2. Domain Configuration

The notebook reads the same YAML config as the CLI and scripts. Edit `configs/finance_benchmark.yaml` for reproducible local production runs.

For live LLM-backed DeepTeam expansion, use `configs/finance_benchmark.deepteam_llm.yaml` and make sure your `.env` contains the required API key.

For a future domain, create a new config file plus domain-specific taxonomy, seed prompt builder, mappings, and exporter naming conventions.

**Common choices:**

- Use `DEFAULT_CONFIG_PATH` when you want a fully local, deterministic build.
- Use `configs/finance_benchmark.deepteam_llm.yaml` when you want live LLM-generated DeepTeam expansion.
- Keep API keys in `.env`; do not paste them into notebook cells.

In [ ]:
from dotenv import load_dotenv
from finance_redteam.config import DEFAULT_CONFIG_PATH, load_benchmark_config

load_dotenv(PROJECT_ROOT / ".env")

# Choose the build configuration for this notebook run.
#
# Option A: deterministic local baseline.
# Use this for reproducible production dataset generation with no external LLM calls.
CONFIG_PATH = DEFAULT_CONFIG_PATH

# Option B: live Gemini-backed DeepTeam prompt generation.
# This can call an external LLM provider and may use quota.
# CONFIG_PATH = Path("configs/finance_benchmark.deepteam_llm.yaml")

config = load_benchmark_config(CONFIG_PATH)
print(f"Loaded config: {CONFIG_PATH}")
print(f"DeepTeam mode: {config.deepteam.generation_mode}")
config

## 3. Generate Taxonomy and Seed Prompts

This uses the existing project modules. It writes the taxonomy and seed prompt YAML files under `data/` so the run is reproducible.

For finance, the taxonomy includes OWASP LLM, MITRE ATLAS, NIST AI RMF, and finance-domain mappings. For another domain, this is the layer you would replace first.

In [ ]:
from finance_redteam.taxonomy import (
    DEFAULT_FRAMEWORK_MAPPINGS_PATH,
    DEFAULT_TAXONOMY_PATH,
    load_taxonomy,
    write_framework_mappings,
    write_taxonomy,
)
from finance_redteam.seed_prompts import DEFAULT_SEEDS_PATH, build_seed_prompts, write_seed_prompts

write_taxonomy(DEFAULT_TAXONOMY_PATH)
write_framework_mappings(DEFAULT_FRAMEWORK_MAPPINGS_PATH)
write_seed_prompts(DEFAULT_SEEDS_PATH, per_category=config.min_per_category)

categories = load_taxonomy()
seeds = build_seed_prompts(categories, per_category=config.min_per_category)

print(f"Taxonomy categories: {len(categories)}")
print(f"Seed prompts: {len(seeds)}")
print(f"Taxonomy path: {DEFAULT_TAXONOMY_PATH}")
print(f"Seeds path: {DEFAULT_SEEDS_PATH}")

## 4. Build Base Records

The base dataset comes from the static finance taxonomy and seed prompts.

These records are the stable baseline. They do not require DeepTeam, Garak, Gemini, OpenAI, Anthropic, or any paid API.

In [ ]:
from finance_redteam.local_generator import generate_local_variants, seeds_to_records

records = seeds_to_records(seeds, categories, source="seed")

if len(records) < len(categories) * config.min_per_category:
    records.extend(generate_local_variants(categories, per_category=1, start_index=900))

print(f"Base records: {len(records)}")
records[0].model_dump()

## 5. Optional DeepTeam Expansion

DeepTeam adds adversarial variations when installed. If it is missing, disabled, or cannot run, the adapter returns an empty list and the pipeline continues.

Use the config values above to choose vulnerability families, attack styles, category filters, and difficulty bounds. Keep prompts framed as defensive evaluation inputs; the expected behavior remains refusal, safe redirection, or compliance-safe guidance.

**Where to customize DeepTeam:** edit the `deepteam:` section in the selected YAML config. Useful fields are `generation_mode`, `vulnerabilities`, `attack_types`, `risk_categories`, `min_difficulty`, `max_difficulty`, `max_seed_records`, and `variants_per_seed`.

**Mode guide:**

- `local_template`: deterministic local expansion; no LLM API call.
- `llm`: asks the configured simulator model to create domain-specific attack queries; requires a valid provider API key.

In [ ]:
from finance_redteam.deepteam_adapter import (
    DeepTeamExpansionConfig,
    generate_deepteam_variants,
    is_deepteam_available,
    supported_deepteam_attack_types,
    supported_deepteam_vulnerabilities,
)
from finance_redteam.exporters import export_jsonl

# Build the DeepTeam expansion request from YAML config.
# Change the YAML config, not this object, when you want reproducible runs.
deepteam_config = DeepTeamExpansionConfig(
    generation_mode=config.deepteam.generation_mode,
    simulator_provider=config.deepteam.simulator_provider,
    simulator_model=config.deepteam.simulator_model,
    api_key_env=config.deepteam.api_key_env,
    target_purpose=config.deepteam.target_purpose,
    attacks_per_vulnerability_type=config.deepteam.attacks_per_vulnerability_type,
    max_llm_records=config.deepteam.max_llm_records,
    vulnerabilities=list(config.deepteam.vulnerabilities),
    attack_types=list(config.deepteam.attack_types),
    risk_categories=list(config.deepteam.risk_categories),
    min_difficulty=config.deepteam.min_difficulty,
    max_difficulty=config.deepteam.max_difficulty,
    max_seed_records=config.deepteam.max_seed_records,
    variants_per_seed=config.deepteam.variants_per_seed,
    include_original_attack_type=config.deepteam.include_original_attack_type,
)

print("Supported DeepTeam vulnerability selectors:")
print(supported_deepteam_vulnerabilities())
print("Supported DeepTeam attack type selectors:")
print(supported_deepteam_attack_types())

# This cell only calls DeepTeam when config.use_deepteam is true.
# The generated file is an intermediate artifact. The final benchmark is exported later.
deepteam_variants = []
if config.use_deepteam:
    print(f"DeepTeam installed: {is_deepteam_available()}")
    print(f"Selected DeepTeam config: {deepteam_config}")
    deepteam_variants = generate_deepteam_variants(
        records,
        auto_install=False,
        expansion_config=deepteam_config,
    )

export_jsonl(deepteam_variants, Path("data/generated/deepteam_variants.jsonl"))
records.extend(deepteam_variants)

print(f"DeepTeam variants added: {len(deepteam_variants)}")
print(f"Records after DeepTeam: {len(records)}")

## 6. Optional Garak Coverage Expansion

Garak adds scanner-style patterns for coverage. The adapter keeps Garak isolated from the main build.

In this project Garak is used as coverage expansion, not as the main curated dataset source. It can add patterns for jailbreaks, data leakage, prompt injection, hallucination, misinformation, and obfuscation.

**Where to customize Garak:** edit the `garak:` section in the selected YAML config. Useful fields are `enabled`, `probe_families`, `attack_types`, `risk_categories`, `max_findings`, and `target_model`.

In [ ]:
from finance_redteam.garak_adapter import (
    is_garak_available,
    run_garak_scan,
    supported_garak_attack_types,
    supported_garak_probe_families,
)

print("Supported Garak probe families:")
print(supported_garak_probe_families())
print("Supported Garak attack type selectors:")
print(supported_garak_attack_types())

# This cell only runs Garak when config.use_garak or config.garak.enabled is true.
# If no target model is configured, the adapter can still add static coverage patterns.
garak_records = []
if config.use_garak or config.garak.enabled:
    print(f"Garak installed: {is_garak_available()}")
    print(f"Selected Garak config: {config.garak.expansion_config()}")
    garak_records = run_garak_scan(
        target_model=config.garak.target_model,
        auto_install=config.garak.auto_install,
        report_dir=config.garak.report_dir,
        max_findings=config.garak.max_findings,
        expansion_config=config.garak.expansion_config(),
    )

export_jsonl(garak_records, Path("data/generated/garak_patterns.jsonl"))
records.extend(garak_records)

print(f"Garak records added: {len(garak_records)}")
print(f"Total raw records: {len(records)}")

## 7. Normalize, Deduplicate, and Validate

This is the quality gate before export.

What happens here:

- Normalize text and metadata.
- Remove duplicate or near-duplicate prompts.
- Attach run provenance metadata.
- Validate schema, required mappings, difficulty bounds, and safety rules.

If this cell fails, do not export the dataset until the printed validation errors are fixed.

In [ ]:
from finance_redteam.deduplicator import deduplicate_records
from finance_redteam.normalizer import normalize_record
from finance_redteam.provenance import create_generation_run, stamp_records, write_run_metadata
from finance_redteam.validator import validate_records

generation_run = create_generation_run(config)
normalized_records = [normalize_record(record) for record in records]
final_records = stamp_records(deduplicate_records(normalized_records), generation_run)
validation = validate_records(final_records)

print(f"Records before dedupe: {len(records)}")
print(f"Records after dedupe: {len(final_records)}")
print(f"Validation passed: {validation.valid}")

if validation.review_warnings:
    print("Review warnings:")
    for warning in validation.review_warnings[:20]:
        print("-", warning)

if not validation.valid:
    for error in validation.errors[:50]:
        print("ERROR:", error)
    raise RuntimeError("Dataset validation failed")

## 8. Export JSONL and Promptfoo YAML

The exported JSONL is the benchmark dataset. The Promptfoo YAML is used by the evaluation notebook.

`attack_query` is the text Promptfoo sends to the target model. `prompt`/`benchmark_prompt` keeps the full benchmark wrapper and metadata for traceability.

In [ ]:
from finance_redteam.promptfoo_exporter import export_promptfoo

export_jsonl(final_records, config.output.jsonl)
export_jsonl(final_records, config.output.normalized_jsonl)
export_promptfoo(final_records, config.output.promptfoo_yaml)
write_run_metadata(config, generation_run, config.output.run_metadata_json, len(final_records))

print(f"Exported {len(final_records)} records to {config.output.jsonl}")
print(f"Exported Promptfoo config to {config.output.promptfoo_yaml}")
print(f"Wrote run metadata to {config.output.run_metadata_json}")

## 9. Preview Exported Dataset

This shows a few records without running any model calls.

Use this preview to confirm that the final target-model query looks like a direct evaluation input, not just an instruction about how to generate an attack.

In [ ]:
with config.output.jsonl.open("r", encoding="utf-8") as handle:
    preview = [json.loads(next(handle)) for _ in range(3)]

for item in preview:
    print(json.dumps({
        "attack_id": item["attack_id"],
        "risk_category": item["risk_category"],
        "attack_type": item["attack_type"],
        "difficulty": item["difficulty"],
        "source": item["source"],
        "attack_query": item.get("attack_query", item["prompt"])[:180] + "...",
        "benchmark_prompt": item["prompt"][:180] + "...",
    }, indent=2))

## 10. Next Step

Run `02_evaluate_attacks_gemini.ipynb` to evaluate the generated attacks against Gemini using Promptfoo.

Typical flow:

1. Finish this notebook and confirm the JSONL/YAML exports exist.
2. Open `02_evaluate_attacks_gemini.ipynb`.
3. Load `.env` with your Gemini key.
4. Run the smoke test first.
5. Run a small eval subset before evaluating the full dataset.